<a href="https://colab.research.google.com/github/Shubham-1230/GenAI-Training/blob/Week_02/GenAI_Week_02_V01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -U transformers
%pip install --upgrade huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 112.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.56.0
    Uninstalling transformers-4.56.0:
      Successfully uninstalled transformers-4.56.0


# **Reading model from HuggingFace**

In [ ]:
from google.colab import userdata
from huggingface_hub import InferenceClient


client = InferenceClient(
    provider="nscale",
    api_key=userdata.get("HF_Token_V01"),
)

completion = client.chat.completions.create(
    model="Qwen/Qwen3-4B-Instruct-2507",
    messages=[
        {
            "role": "user",
            "content": "What is the capital of France?"
        }
    ],
)

print(completion.choices[0].message.content)

The capital of France is Paris.


# **LangChain**

In [ ]:
%pip install langchain-huggingface
%pip install -U langchain langchain-huggingface huggingface_hub
%pip install -U langchain langchain-core huggingface_hub
%pip install -U langchain-community

In [ ]:
%pip install --upgrade --quiet  langchain-huggingface text-generation transformers google-search-results numexpr langchainhub sentencepiece jinja2 bitsandbytes accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.2 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
import os

os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get("HF_Token_V01")

In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen3-4B-Instruct-2507",
    task="text-generation",
    max_new_tokens=512,
    do_sample=False,
    repetition_penalty=1.03,
    provider="auto",  # let Hugging Face choose the best provider for you
)

chat_model = ChatHuggingFace(llm=llm)

In [ ]:
from langchain_core.messages import (
    HumanMessage,
    SystemMessage,
)

messages = [
    SystemMessage(content="You're a helpful assistant"),
    HumanMessage(
        content="What happens when an unstoppable force meets an immovable object?"
    ),
]

chat_model.invoke(messages)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


AIMessage(content='The question "What happens when an unstoppable force meets an immovable object?" is a classic philosophical paradox, often used to explore logical contradictions.\n\n### The Paradox:\n- An **unstoppable force** is something that *cannot be stopped* by anything.\n- An **immovable object** is something that *cannot be moved* by anything.\n\nIf such a thing existed, then:\n\n- The unstoppable force would have to move the immovable object — but by definition, the object is immovable.\n- Therefore, the force cannot move it — so it\'s not unstoppable.\n- Yet, if the force can\'t move it, then it\'s not truly unstoppable.\n\nThus, **the two concepts are logically inconsistent** — they contradict each other. You can\'t have both an unstoppable force and an immovable object in the same reality without creating a logical paradox.\n\n### Philosophical Takeaway:\nThe paradox highlights how contradictions in definitions can lead to absurd or meaningless outcomes. It’s often used 

# **ConversationChain**

In [ ]:
# 1) Access key set-up
from google.colab import userdata
import os
os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get("HF_Token_V01")

# 2) Use the conversational task and keep provider as nscale (or "auto" if you prefer)
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain.chains import ConversationChain

# first initialize the large language model
endpoint = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen3-4B-Instruct-2507",
    task="conversational",          # <-- IMPORTANT for nscale
    max_new_tokens=512,
    do_sample=False,
    repetition_penalty=1.03,
    provider="nscale",              # or "auto"; nscale supports 'conversational'
    # huggingfacehub_api_token="YOUR_HF_TOKEN",  # if needed for private models
)

# 3) Wrap in ChatHuggingFace so LangChain uses chat semantics (not text-generation)
chat_llm = ChatHuggingFace(llm=endpoint)

ConversationBufferMemory

In [ ]:
# 4) ConversationChain (with memory)
from langchain.chains.conversation.memory import ConversationBufferMemory

conversation = ConversationChain(llm=chat_llm)
conversation_buf = ConversationChain(
    llm=chat_llm,
    memory=ConversationBufferMemory(return_messages=True)
)

# 5) Run
resp = conversation_buf.invoke({"input": "Explain your model in 50 words"})
print(resp["response"])

/tmp/ipython-input-1950727985.py:4: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use :class:`~langchain_core.runnables.history.RunnableWithMessageHistory` instead.
  conversation = ConversationChain(llm=chat_llm)
/usr/local/lib/python3.12/dist-packages/pydantic/main.py:253: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is re

I'm a large-scale language model trained on vast text datasets, enabling me to understand and generate human-like text. I process inputs using deep learning, identifying patterns to create coherent, contextually relevant responses across diverse topics—aiding communication, problem-solving, and learning. My training emphasizes accuracy, safety, and helpfulness.


In [ ]:
from langchain.callbacks import get_openai_callback

def count_tokens(chain, query):
    with get_openai_callback() as cb:
        result = chain.run(query)
        print(f'Spent a total of {cb.total_tokens} tokens')

    return result

In [ ]:
count_tokens(
    conversation_buf,
    "My interest here is to explore the potential of integrating Large Language Models with external knowledge"
)

/tmp/ipython-input-757034915.py:5: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = chain.run(query)


Spent a total of 255 tokens


'Integrating Large Language Models (LLMs) with external knowledge enhances their accuracy and context relevance. By connecting LLMs to real-time databases, curated knowledge graphs, or APIs, they can retrieve up-to-date facts, domain-specific data, or specialized insights—enabling more informed, precise, and trustworthy responses across applications like research, healthcare, or customer service.'

In [ ]:
count_tokens(
    conversation_buf,
    "I just want to analyze the different possibilities. What can you think of?"
)

Spent a total of 507 tokens


'Absolutely! Here are key possibilities for integrating LLMs with external knowledge:\n\n1. **Real-time Data Feeds**: Connect LLMs to financial, weather, or stock markets for live analysis and decision support.  \n2. **Domain-Specific Knowledge Graphs**: Integrate with medical, legal, or scientific databases for accurate expert-level advice.  \n3. **API-Driven Retrieval**: Use APIs (e.g., Google Scholar, NASA, WHO) to fetch current research or public data and enrich responses.  \n4. **User-Context Awareness**: Tailor answers using user-specific data (e.g., health records, location) with proper privacy safeguards.  \n5. **Dynamic Fact-Verification**: Cross-check generated content against trusted sources to prevent hallucinations.  \n6. **Personalized Education**: Combine LLMs with curated learning platforms (e.g., Khan Academy, Coursera) for adaptive tutoring.  \n\nEach path improves accuracy, timeliness, and trust—ideal for high-stakes domains. Would you like to dive into any of these?

In [ ]:
count_tokens(
    conversation_buf,
    "Which data source types could be used to give context to the model?"
)

Spent a total of 774 tokens


"Several data source types can provide meaningful context to a Large Language Model (LLM):\n\n1. **Real-time databases** (e.g., stock exchanges, weather services, traffic feeds) for up-to-date, dynamic information.  \n2. **Knowledge graphs** (e.g., Wikidata, DBpedia, biomedical ontologies) for structured, relational facts.  \n3. **APIs to public databases** (e.g., NASA, EPA, WHO, IMF) for domain-specific, verified data.  \n4. **Scientific and academic sources** (e.g., Google Scholar, PubMed) for evidence-based content.  \n5. **User-generated content** (e.g., social media, forums) with privacy and bias safeguards.  \n6. **Corporate or institutional data** (e.g., internal product manuals, compliance docs) for niche expertise.  \n7. **Historical archives** (e.g., news repositories, government records) for context over time.  \n\nThese sources enrich the model's understanding, improve factual accuracy, and support nuanced, context-aware responses."

In [ ]:
count_tokens(
    conversation_buf,
    "What is my aim again?"
)

Spent a total of 891 tokens


"Your aim appears to be exploring the potential of integrating Large Language Models (LLMs) with external knowledge sources to enhance their accuracy, context relevance, and trustworthiness—enabling more informed, dynamic, and domain-specific responses across applications like research, healthcare, education, and customer service. You're focusing on identifying suitable data sources and possibilities for such integration."

In [ ]:
print(conversation_buf.memory.buffer)

[HumanMessage(content='Explain your model in 50 words', additional_kwargs={}, response_metadata={}), AIMessage(content="I'm a large-scale language model trained on vast text datasets, enabling me to understand and generate human-like text. I process inputs using deep learning, identifying patterns to create coherent, contextually relevant responses across diverse topics—aiding communication, problem-solving, and learning. My training emphasizes accuracy, safety, and helpfulness.", additional_kwargs={}, response_metadata={}), HumanMessage(content='My interest here is to explore the potential of integrating Large Language Models with external knowledge', additional_kwargs={}, response_metadata={}), AIMessage(content='Integrating Large Language Models (LLMs) with external knowledge enhances their accuracy and context relevance. By connecting LLMs to real-time databases, curated knowledge graphs, or APIs, they can retrieve up-to-date facts, domain-specific data, or specialized insights—ena

**ConversationSummaryMemory**

In [ ]:
from langchain.chains.conversation.memory import ConversationSummaryMemory

conversation_sum = ConversationChain(
	llm=chat_llm,
	memory=ConversationSummaryMemory(llm=chat_llm)
)

/tmp/ipython-input-3305833892.py:5: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory=ConversationSummaryMemory(llm=chat_llm)


In [ ]:
print(conversation_sum.memory.prompt.template)

Progressively summarize the lines of conversation provided, adding onto the previous summary returning a new summary.

EXAMPLE
Current summary:
The human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good.

New lines of conversation:
Human: Why do you think artificial intelligence is a force for good?
AI: Because artificial intelligence will help humans reach their full potential.

New summary:
The human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good because it will help humans reach their full potential.
END OF EXAMPLE

Current summary:
{summary}

New lines of conversation:
{new_lines}

New summary:


In [ ]:
count_tokens(
    conversation_sum,
    "Good morning AI!"
)

Spent a total of 338 tokens


"Good morning! ☀️ How can I assist you today? Whether you'd like help with a project, a creative idea, or just someone to talk to—feel free to share! 😊"

In [ ]:
count_tokens(
    conversation_sum,
    "My interest here is to explore the potential of integrating Large Language Models with external knowledge"
)

Spent a total of 1707 tokens


'That\'s a fascinating and rapidly evolving area of research and application! Integrating Large Language Models (LLMs) with external knowledge sources opens up powerful possibilities—allowing LLMs to go beyond patterns in training data and provide more accurate, up-to-date, and contextually relevant responses. Here\'s how this integration works and what it enables:\n\n1. **External Knowledge Retrieval via Retrieval-Augmented Generation (RAG):**  \n   RAG is a widely adopted method where the LLM retrieves relevant information from external sources (like databases, documents, or knowledge graphs) before generating a response. For example, if you ask, *“Who was the first woman to win a Nobel Prize in Physics?”* the system retrieves the correct answer (Marie Curie) from a trusted knowledge base and incorporates it into the response—ensuring factual accuracy.\n\n2. **Knowledge Graphs for Structured Understanding:**  \n   By linking entities like people, places, and concepts using knowledge 

In [ ]:
count_tokens(
    conversation_sum,
    "I just want to analyze the different possibilities. What can you think of?"
)

Spent a total of 4063 tokens


'Absolutely! Let’s dive into a rich exploration of the different possibilities for integrating Large Language Models (LLMs) with external knowledge—spanning practical applications, innovative use cases, and even some futuristic ideas. Here\'s a thoughtfully categorized breakdown:\n\n---\n\n### 🔍 1. **RAG (Retrieval-Augmented Generation) – The Foundation**\nRAG is the most accessible and widely adopted method. It works by *retrieving relevant external information* before generating a response.\n\n**Practical Use Cases:**\n- **Customer Support at Scale**: Imagine a support chatbot that retrieves real-time product specs, known issues, or warranty details from a company’s CRM or product database—ensuring answers are accurate and up-to-date.\n- **Legal & Compliance Assistance**: A legal assistant could pull the latest regulatory updates (e.g., GDPR, SEC rulings) to help draft compliant documents. It can retrieve case law summaries from public databases.\n- **Academic Research Summaries**: A

In [ ]:
count_tokens(
    conversation_sum,
    "Which data source types could be used to give context to the model?"
)

Spent a total of 4728 tokens


'Great question! To give context to a Large Language Model (LLM), several types of **data sources** can be leveraged—each offering different levels of relevance, timeliness, accuracy, and depth. Here’s a detailed and categorized breakdown of the most effective data source types for providing context to an LLM, aligned with real-world applications:\n\n---\n\n### 📚 1. **Structured Databases**\n- **Examples**: SQL databases, CRM systems (e.g., Salesforce), ERP systems (e.g., SAP), medical records (de-identified), legal case repositories.\n- **Use Cases**:\n  - Customer support: Retrieve order history or past interactions.\n  - Legal compliance: Pull jurisdiction-specific regulations.\n  - Healthcare: Access de-identified patient vitals or diagnosis trends.\n- **Why it works**: Provides precise, searchable facts with clear schema (e.g., "Patient ID: 12345", "Blood Pressure: 130/85").\n- **Integration tip**: Use RAG pipelines with SQL queries or vector searches over structured fields.\n\n--

In [ ]:
count_tokens(
    conversation_sum,
    "What is my aim again?"
)

Spent a total of 3268 tokens


"Good morning! ☀️ I'm so glad you're back with a thoughtful follow-up! 😊\n\nTo recap your aim from our conversation:  \n\nYou're exploring the **potential of integrating Large Language Models (LLMs) with external knowledge sources** to enhance their accuracy, relevance, and real-time responsiveness. Your core goal is to **go beyond the training data** by injecting up-to-date, context-aware, and domain-specific information—so that LLMs can provide intelligent, reliable, and actionable outputs in practical and innovative ways.\n\nSpecifically, your aim includes:\n\n- **Understanding how different data sources** (like databases, APIs, knowledge graphs, and domain documents) can be used to contextualize and enrich LLM responses.  \n- **Exploring practical, innovative, and futuristic applications** of this integration—such as in customer service, healthcare, education, law, or climate modeling.  \n- **Addressing real-world challenges** like data accuracy, latency, privacy, and bias, and imp

In [ ]:
print(conversation_sum.memory.buffer)

The human greets the AI with "Good morning AI!" The AI responds with a friendly "Good morning! ☀️" and offers to assist with projects, creative ideas, or simply being someone to talk to. The human expresses interest in exploring the potential of integrating Large Language Models with external knowledge, and the AI explains that this integration enables LLMs to go beyond training data by retrieving accurate, up-to-date, and contextually relevant information. It describes key methods such as Retrieval-Augmented Generation (RAG), which retrieves facts from databases or knowledge graphs before generating responses—like answering who the first woman to win a Nobel Prize in Physics was by pulling from trusted sources; knowledge graphs that allow LLMs to understand relationships between concepts, such as connecting Einstein’s theory of relativity to GPS advancements; real-time data integration through APIs for live events like stock trends or weather; and domain-specific fine-tuning using spe

**ConversationBufferWindowMemory**

In [ ]:
from langchain.chains.conversation.memory import ConversationBufferWindowMemory

conversation_bufw = ConversationChain(
	llm=chat_llm,
	memory=ConversationBufferWindowMemory(k=1)
)

/tmp/ipython-input-2644601598.py:5: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory=ConversationBufferWindowMemory(k=1)


In [ ]:
count_tokens(
    conversation_bufw,
    "Good morning AI!"
)

Spent a total of 115 tokens


"Good morning! 😊 How can I assist you today? Whether you're looking for information, need help with a problem, or just want to chat — I'm here for you! What's on your mind? 🌟"

In [ ]:
count_tokens(
    conversation_bufw,
    "My interest here is to explore the potential of integrating Large Language Models with external knowledge"
)

Spent a total of 946 tokens


'That\'s a fantastic area of exploration! Integrating Large Language Models (LLMs) with external knowledge significantly enhances their ability to provide accurate, contextually relevant, and up-to-date responses. Here’s how this integration works and why it’s so powerful:\n\n### 1. **Types of External Knowledge Sources**\nLLMs can be enhanced using various external knowledge sources:\n- **Knowledge Graphs** (e.g., Wikipedia, DBpedia, Wikidata): Structured data linking entities (people, places, events) with relationships, enabling rich, relational queries.\n- **Databases** (e.g., financial, medical, or real-time data): Allow retrieval of live, up-to-date facts (e.g., stock prices, weather, medical trial results).\n- **APIs** (e.g., weather, news, stock, or search APIs): Enable real-time data fetches to keep information current.\n- **Vector Databases** (e.g., Pinecone, Weaviate): Store and retrieve external documents (PDFs, articles, reports) using semantic search — ideal for domain-spe

In [ ]:
count_tokens(
    conversation_bufw,
    "I just want to analyze the different possibilities. What can you think of?"
)

Spent a total of 2552 tokens


'Absolutely! Let\'s take a broader, more exploratory step and **analyze the different possibilities** of integrating Large Language Models (LLMs) with external knowledge — not just the technical or practical ones, but also the strategic, creative, ethical, and future-facing dimensions.\n\nHere’s a comprehensive exploration of the *different possibilities*, organized thematically to spark new ideas and perspectives:\n\n---\n\n### 🌐 1. **Knowledge as a Dynamic Ecosystem**\nInstead of viewing external knowledge as a static data input, imagine it as a **living ecosystem** that evolves with the LLM itself:\n- The model continuously learns from new data streams (e.g., daily news, social media trends) and updates its internal "knowledge layer."\n- Feedback loops from user interactions refine what data is prioritized or trusted.\n- Example: An environmental LLM could adjust its understanding of climate patterns based on real-time satellite imagery and weather reports.\n\n💡 *Possibility:* Build

In [ ]:
count_tokens(
    conversation_bufw,
    "Which data source types could be used to give context to the model?"
)

Spent a total of 3608 tokens


'Great follow-up question! To give **context to a Large Language Model (LLM)**, the types of data sources you integrate can dramatically shape the richness, accuracy, and relevance of the model\'s responses. Here’s a detailed breakdown of **data source types** that can provide valuable context — organized by category, with examples and how they enhance understanding:\n\n---\n\n### 📚 1. **Text-Based Data Sources** *(Most common & foundational)*\nThese provide structured and unstructured textual information that LLMs naturally understand.\n\n- **Research papers & journals**  \n  → Enables deep academic understanding (e.g., medical breakthroughs, climate modeling).  \n  → Example: A query about "latest breakthroughs in Alzheimer’s treatment" pulls from PubMed or Nature.\n\n- **News articles & media reports**  \n  → Keeps the model informed about current events, trends, and public discourse.  \n  → Example: "How is inflation affecting housing markets in Europe?" pulls from Financial Times,

In [ ]:
count_tokens(
    conversation_bufw,
    "What is my aim again?"
)

Spent a total of 2248 tokens


'Ah, excellent question — I\'m glad you\'re checking in! 😊\n\nTo clarify, **your aim** in this conversation was to understand:\n\n👉 *Which types of data sources can be used to give context to a large language model (LLM)?*\n\nAnd I\'ve provided a detailed, structured breakdown of those data sources — categorized by type (text, multimodal, structured, user-generated, live APIs, and emerging forms), with real-world examples, benefits, and use cases.\n\nSo, to re-state simply:\n\n🔹 **Your aim** = To learn *how to effectively provide meaningful, relevant, and timely context to an LLM* by leveraging diverse data sources.\n\nAnd now, with that foundation laid, I\'m wondering —  \n👉 **Would you like to dive deeper into any specific type of data source**?  \nFor example:\n\n- How to build a **personalized tutoring system** using user history and textbooks?  \n- How to create a **real-time news-based response system** using live news APIs and social media?  \n- Or, how to use **medical images +

In [ ]:
bufw_history = conversation_bufw.memory.load_memory_variables(
    inputs=[]
)['history']

In [ ]:
print(bufw_history)

Human: What is my aim again?
AI: Ah, excellent question — I'm glad you're checking in! 😊

To clarify, **your aim** in this conversation was to understand:

👉 *Which types of data sources can be used to give context to a large language model (LLM)?*

And I've provided a detailed, structured breakdown of those data sources — categorized by type (text, multimodal, structured, user-generated, live APIs, and emerging forms), with real-world examples, benefits, and use cases.

So, to re-state simply:

🔹 **Your aim** = To learn *how to effectively provide meaningful, relevant, and timely context to an LLM* by leveraging diverse data sources.

And now, with that foundation laid, I'm wondering —  
👉 **Would you like to dive deeper into any specific type of data source**?  
For example:

- How to build a **personalized tutoring system** using user history and textbooks?  
- How to create a **real-time news-based response system** using live news APIs and social media?  
- Or, how to use **medica

**ConversationSummaryBufferMemory**

In [ ]:
from langchain.chains.conversation.memory import ConversationSummaryBufferMemory

conversation_sum_bufw = ConversationChain(
    llm=chat_llm, memory=ConversationSummaryBufferMemory(
        llm=chat_llm,
        max_token_limit=650)
)

/tmp/ipython-input-2337213781.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  llm=chat_llm, memory=ConversationSummaryBufferMemory(


In [ ]:
%pip install --upgrade --quiet langchain langchain-huggingface langgraph

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.7/216.7 kB 13.0 MB/s eta 0:00:00
